In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "sensor_id":   ["A","A","A","B","B","B","C","C"],
    "ts":          pd.to_datetime(["2026-01-01 00:00","2026-01-01 00:00","2026-01-01 01:00",
                                   "2026-01-01 00:00","2026-01-01 00:00","2026-01-01 01:00",
                                   "2026-01-01 00:00","2026-01-01 00:00"]),
    "value":       [10.1, 10.1, 10.4, 7.0, 7.3, 7.1, np.nan, 5.5],
    "ingested_at": pd.to_datetime(["2026-01-01 00:05","2026-01-01 00:05","2026-01-01 01:05",
                                   "2026-01-01 00:05","2026-01-01 02:00","2026-01-01 01:05",
                                   "2026-01-01 00:05","2026-01-01 03:00"]),
})

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   sensor_id    8 non-null      str           
 1   ts           8 non-null      datetime64[us]
 2   value        7 non-null      float64       
 3   ingested_at  8 non-null      datetime64[us]
dtypes: datetime64[us](2), float64(1), str(1)
memory usage: 388.0 bytes


In [ ]:
df_clean = df.copy()

ndups = df_clean.duplicated().sum()
print(f"num of dups: {ndups}")
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

check = df_clean.duplicated(subset=['sensor_id', 'ts'], keep=False)
print(df_clean[check])

df_clean = (
    df_clean
    .sort_values("ingested_at")
    .drop_duplicates(subset=['sensor_id', 'ts'], keep='last')
    .sort_values(by = ['sensor_id', 'ts'])
    .reset_index(drop=True)
)
# print(df_clean)


num of dups: 1
  sensor_id         ts  value         ingested_at
2         B 2026-01-01    7.0 2026-01-01 00:05:00
3         B 2026-01-01    7.3 2026-01-01 02:00:00
5         C 2026-01-01    NaN 2026-01-01 00:05:00
6         C 2026-01-01    5.5 2026-01-01 03:00:00


,sensor_id,ts,value,ingested_at
0,A,2026-01-01 00:00:00,10.1,2026-01-01 00:05:00
1,A,2026-01-01 01:00:00,10.4,2026-01-01 01:05:00
2,B,2026-01-01 00:00:00,7.3,2026-01-01 02:00:00
3,B,2026-01-01 01:00:00,7.1,2026-01-01 01:05:00
4,C,2026-01-01 00:00:00,5.5,2026-01-01 03:00:00


In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(7)
dates = pd.date_range("2026-01-05", periods=56, freq="D")      # starts on a Monday
weekly = np.array([0, 20, 30, 25, 10, -60, -90])                 # Mon..Sun pattern
y = 1000 + np.tile(weekly, 8) + 1.5 * np.arange(56) + rng.normal(0, 12, 56)
s = pd.Series(y.round(1), index=dates, name="throughput_t")

In [4]:
from sklearn.metrics import mean_absolute_error
# 6 weeks -> 42 days
n = 42

train, test = s.iloc[:n ], s.iloc[n:]

mean_pred = np.full(len(test), np.mean(train))
last_val_pred = np.full(len(test), train.tolist()[-1])
seasonal_pred = train[n-7:].tolist() * 2

preds = {'mean': mean_pred, "last_value": last_val_pred, "seasonal_naive": seasonal_pred}

res = pd.DataFrame({k: [mean_absolute_error(test, v)] for k, v in preds.items()},
                   index=["MAE"]).T.sort_values(by="MAE", ascending=True)

res

,MAE
seasonal_naive,18.742857
mean,57.071429
last_value,92.614286


In [1]:
import pandas as pd, numpy as np
df = pd.DataFrame({
    "machine":   ["M1","M1","M1","M1","M2","M2","M2","M2","M3","M3","M3","M3"],
    "temp":      [70, 72, 71, 69, 85, 88, 90, 86, 71, 70, 73, 72],
    "viscosity": [1.2, 1.3, np.nan, 1.1, np.nan, np.nan, 2.4, np.nan, 1.5, 1.4, 1.6, np.nan],
    "operator":  ["a","b","a",None,"c","c","d","c","e","e",None,"f"],
})

In [26]:
df_clean = df.copy()
print(df_clean.isna().mean().mul(100).round(1))

df['viscosity'].isna().groupby(df['machine']).mean()

print(df_clean.groupby('machine')['viscosity'].apply(lambda x: x.isna().mean()))
# # test 
# df_clean['visc_missing'] = df_clean['viscosity'].isna()
# print(df_clean.groupby('visc_missing')['temp'].mean().round(2))
# df_clean['viscosity'] = df_clean.groupby('machine')['viscosity'].transform(lambda x: x.fillna(value=x.median()))
# print(df_clean)

machine       0.0
temp          0.0
viscosity    41.7
operator     16.7
dtype: float64
machine
M1    0.25
M2    0.75
M3    0.25
Name: viscosity, dtype: float64


In [1]:
import numpy as np, pandas as pd
rng = np.random.default_rng(3)
n = 400
temp = rng.normal(80, 5, n)
pressure = rng.normal(2.0, 0.3, n)
feed_rate = rng.normal(300, 30, n)
logit = -1.5 + 0.15*(temp-80) - 1.2*(pressure-2.0) + 0.01*(feed_rate-300)
failed = (rng.random(n) < 1/(1+np.exp(-logit))).astype(int)
df = pd.DataFrame({
    "batch_id": np.arange(1001, 1001+n),
    "temp": temp.round(1),
    "pressure": pressure.round(2),
    "feed_rate": feed_rate.round(0),
    "scrap_kg": np.where(failed==1, rng.gamma(4, 20, n), rng.gamma(1, 2, n)).round(1),
    "rework_hours": np.where(failed==1, rng.uniform(1, 8, n), 0).round(1),
    "failed": failed,
})

In [4]:
print(df['failed'].mean())

0.2


In [10]:
from sklearn.metrics import roc_auc_score

feats = df.columns.drop("failed")
auc = pd.Series({c: roc_auc_score(df['failed'], df[c]) for c in feats})
screen = (pd.DataFrame({"auc": auc, "strength" : np.maximum(auc, 1-auc)}).sort_values("strength", ascending = False).round(3))

screen

,auc,strength
scrap_kg,1.000,1.000
rework_hours,1.000,1.000
temp,0.650,0.650
feed_rate,0.630,0.630
pressure,0.395,0.605
batch_id,0.524,0.524


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = df.drop(columns = 'failed'), df['failed'] 
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)

ALL = list(feats)
CLEAN = ['temp', 'pressure', 'feed_rate']

for name, cols in [('all', ALL), ('clean', CLEAN)]:

    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    model.fit(X_tr[cols], y_tr)
    proba = model.predict_proba(X_te[cols])[:, 1]

    print(f"{name:5s} AUC: {roc_auc_score(y_te, proba):.3f}")

all   AUC: 1.000
clean AUC: 0.744


In [21]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

model_dum = make_pipeline(StandardScaler(), DummyClassifier(strategy='prior'))
model_logi = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model_hgb = make_pipeline(StandardScaler(), HistGradientBoostingClassifier(random_state=0))


res = {"model_name": [], "auc": []}
for model_name, model in [("dummy", model_dum),  ('Hgb', model_hgb),  ('Logi', model_logi)]:
    model.fit(X_tr[CLEAN], y_tr)
    proba = model.predict_proba(X_te[CLEAN])[:,1]

    res["model_name"].append(model_name)
    res["auc"].append(roc_auc_score(y_te, proba))


res_df = pd.DataFrame(res).sort_values(by='auc', ascending=False)

res_df




,model_name,auc
2,Logi,0.744375
1,Hgb,0.659062
0,dummy,0.500000


In [24]:
import pandas as pd
df = pd.DataFrame({
    "line":   ["L1","L1","L1","L2","L2","L2","L3","L3"],
    "day":    [1,2,3,1,2,3,1,2],
    "tonnes": [100,120,110,80,95,85,150,130],
})

In [45]:
df_agg = df.groupby('line', as_index=False).agg(avg_t = ('tonnes', 'mean'), max_t = ('tonnes', 'max'), n_days = ('day', 'count'))
df_agg
df_row = df.assign(
    line_avg = lambda x: x.groupby('line')['tonnes'].transform('mean').round(2),
    dev = lambda x: x.tonnes - x.line_avg,
    share_pct = lambda x: (x.tonnes / x.groupby('line')['tonnes'].transform('sum')* 100 ).round(1)  ,
    best_day = lambda x: x.tonnes >= x.groupby('line')['tonnes'].transform('max')
)
df_row

,line,day,tonnes,line_avg,dev,share_pct,best_day
0,L1,1,100,110.00,-10.00,30.3,False
1,L1,2,120,110.00,10.00,36.4,True
2,L1,3,110,110.00,0.00,33.3,False
3,L2,1,80,86.67,-6.67,30.8,False
4,L2,2,95,86.67,8.33,36.5,True
5,L2,3,85,86.67,-1.67,32.7,False
6,L3,1,150,140.00,10.00,53.6,True
7,L3,2,130,140.00,-10.00,46.4,False


In [46]:
import numpy as np, pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=2000, n_features=6, n_informative=4,
                           weights=[0.95], flip_y=0.01, class_sep=0.8, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]

In [84]:
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

# 1. 
print(y_te.mean(), y_te.sum(), 1-y_te.mean())

# 2. 
print(f"ROC-AUC: {roc_auc_score(y_te, proba)}, PR-AUC: {average_precision_score(y_te, proba)}")

# 3. 
pred = model.predict(X_te)
confusion_matrix(y_te, pred, labels=[0, 1])
tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel().tolist()
# print((tn, fp, fn, tp))
acc = (tn + tp) / (tn + fp + fn + tp)
recall = tp / (tp + fn)

# 4.
res = {'threshold':[], 'cost': []}
for th in np.round(np.arange(0.05, 0.96, 0.05), 2):
    pred = proba >= th
    tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel().tolist()
    print((tn, fp, fn, tp))
    cost = 50*fp + 500 * fn
    res['threshold'].append(th)
    res['cost'].append(cost)


pd.DataFrame(res).sort_values('cost')
    


0.051666666666666666 31 0.9483333333333334
ROC-AUC: 0.7998185838199444, PR-AUC: 0.1907339869505411
(402, 167, 9, 22)
(500, 69, 17, 14)
(535, 34, 22, 9)
(552, 17, 27, 4)
(562, 7, 29, 2)
(566, 3, 29, 2)
(568, 1, 30, 1)
(568, 1, 31, 0)
(568, 1, 31, 0)
(568, 1, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)
(569, 0, 31, 0)


,threshold,cost
1,0.10,11950
2,0.15,12700
0,0.05,12850
3,0.20,14350
5,0.30,14650
4,0.25,14850
6,0.35,15050
11,0.60,15500
14,0.75,15500
10,0.55,15500


In [85]:
import pandas as pd
ts_a = pd.date_range("2026-03-01 00:00", periods=8, freq="h")
ts_b = ts_a.delete(3)                        # machine B is missing one hour
df = pd.DataFrame({
    "ts": list(ts_a) + list(ts_b),
    "machine": ["A"]*8 + ["B"]*7,
    "vib": [1.0,1.2,1.1,1.5,1.4,1.6,2.0,2.2,  3.0,3.1,2.9,3.3,3.2,3.6,3.8],
}).sample(frac=1, random_state=0).reset_index(drop=True)bbb

In [109]:
df

,ts,machine,vib
0,2026-03-01 01:00:00,A,1.2
1,2026-03-01 06:00:00,A,2.0
2,2026-03-01 00:00:00,B,3.0
3,2026-03-01 01:00:00,B,3.1
4,2026-03-01 07:00:00,B,3.8
5,2026-03-01 04:00:00,A,1.4
6,2026-03-01 02:00:00,A,1.1
7,2026-03-01 06:00:00,B,3.6
8,2026-03-01 02:00:00,B,2.9
9,2026-03-01 07:00:00,A,2.2


In [132]:
w = df.pivot(index = 'ts', columns = 'machine', values = 'vib').asfreq('h')
w[w.isna().any(axis=1)]
#
df2 = df.copy().sort_values(['machine', 'ts'])
df2['lag1'] = df2.groupby('machine')['vib'].transform('shift')
df2['roll3_past'] = df2.groupby('machine')['lag1'].transform(lambda x: x.rolling(3, min_periods = 1 ).mean())
df2
#
# df_clean['roll3_past'] = df_clean['lag1'].rolling(3).mean()
# df_clean['h'] = df['ts'].asfreq('1h')
# df_clean

df.set_index("ts").groupby("machine")["vib"].resample("4h").mean().unstack(0)
df

,ts,machine,vib
0,2026-03-01 01:00:00,A,1.2
1,2026-03-01 06:00:00,A,2.0
2,2026-03-01 00:00:00,B,3.0
3,2026-03-01 01:00:00,B,3.1
4,2026-03-01 07:00:00,B,3.8
5,2026-03-01 04:00:00,A,1.4
6,2026-03-01 02:00:00,A,1.1
7,2026-03-01 06:00:00,B,3.6
8,2026-03-01 02:00:00,B,2.9
9,2026-03-01 07:00:00,A,2.2


In [133]:
import numpy as np, pandas as pd
rng = np.random.default_rng(5)
n = 300
df = pd.DataFrame({
    "tonnage":  rng.normal(500, 80, n).round(0),
    "temp_c":   rng.normal(30, 4, n).round(1),
    "ore_type": rng.choice(["oxide", "sulfide", "mixed"], n, p=[0.3, 0.5, 0.2]),
    "shift":    rng.choice(["Day", "Night"], n),
})
df["energy_mwh"] = (0.02*df["tonnage"] + 0.3*df["temp_c"]
                    + df["ore_type"].map({"oxide": 2, "sulfide": 5, "mixed": 3.5})
                    + (df["shift"] == "Night")*1.0 + rng.normal(0, 1.0, n)).round(2)
df.loc[rng.choice(n, 25, replace=False), "tonnage"] = np.nan
df.loc[rng.choice(n, 15, replace=False), "ore_type"] = np.nan

new_rows = pd.DataFrame({"tonnage": [520, np.nan], "temp_c": [31.0, 28.5],
                         "ore_type": ["laterite", "oxide"], "shift": ["Day", "Night"]})

In [136]:
df

,tonnage,temp_c,ore_type,shift,energy_mwh
0,NaN,25.7,sulfide,Day,22.30
1,NaN,34.8,oxide,Night,19.88
2,480.0,29.8,sulfide,Night,23.97
3,534.0,27.2,oxide,Night,22.83
4,591.0,27.4,oxide,Night,24.54
...,...,...,...,...,...
295,513.0,33.1,sulfide,Day,22.85
296,560.0,25.1,oxide,Day,21.74
297,452.0,31.0,mixed,Day,22.96
298,379.0,34.7,oxide,Night,19.31


In [198]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

num_cols, cat_cols = ['tonnage', 'temp_c'], ['ore_type', 'shift']
X, y = df.drop('energy_mwh', axis =1), df['energy_mwh']

num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler())
])

cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

pre = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])
model = Pipeline([('preproc', pre), ('model', Ridge(alpha=1.0))])

## normal train-test
# X_tr, X_te, y_tr, y_te = train_test_split(
#     X, y, test_size=0.2, random_state=43
# )
# model.fit(X_tr, y_tr)
# y_pred = model.predict(X_te)

# ridge_mae = mean_absolute_error(y_te, y_pred)
# ridge_r2 = r2_score(y_te, y_pred)

# dum_model = DummyRegressor(strategy='mean')
# dum_model.fit(X_tr, y_tr)
# dum_pred = dum_model.predict(X_te)
# dum_mae = mean_absolute_error(y_te, dum_pred)
# dum_r2 = r2_score(y_te, dum_pred)

## cv 
cvfold = KFold(5, shuffle=True, random_state=43)

for name, m in [('dummy', DummyRegressor()), ('ridge', model)]:
    mae = -cross_val_score(m, X, y, cv=cvfold, scoring='neg_mean_absolute_error')
    # cross_val_score(m, X, y, cv = cv, scoring='neg_mean_absolute_error')
    print(f"{name}: mae: {mae.mean().round(2)} +- {mae.std().round(2)}")

model.fit(X, y)
model.predict(new_rows).round(2)


# res_df = pd.DataFrame({
#     'model': ['dummy', 'ridge'],
#     'mae': [dum_mae, ridge_mae],
#     'r2': [dum_r2, ridge_r2]
# })

# res_df



dummy: mae: 2.07 +- 0.17
ridge: mae: 0.83 +- 0.07


array([23.24, 21.5 ])

In [177]:
ridge_r2

0.7480870451479689

In [199]:
import numpy as np, pandas as pd
rng = np.random.default_rng(22)
n = 2000
df = pd.DataFrame({
    "pump_id": rng.integers(1, 150, n),
    "age_years": rng.uniform(0, 15, n).round(1),
    "vibration_mm_s": rng.gamma(3, 1.2, n).round(2),
    "bearing_temp_c": rng.normal(65, 8, n).round(1),
    "hours_since_service": rng.uniform(0, 4000, n).round(0),
    "pump_model": rng.choice(["PX100", "PX200", "QZ9"], n, p=[0.5, 0.35, 0.15]),
})
logit = (-4.2 + 0.12*df["age_years"] + 0.35*(df["vibration_mm_s"] - 3.6)
         + 0.06*(df["bearing_temp_c"] - 65) + 0.0004*df["hours_since_service"]
         + df["pump_model"].map({"PX100": 0, "PX200": 0.3, "QZ9": 0.9}))
df["failed_7d"] = (rng.random(n) < 1/(1 + np.exp(-logit))).astype(int)
df["repair_ticket"] = np.where(df["failed_7d"] == 1, rng.random(n) < 0.95,
                               rng.random(n) < 0.02).astype(int)
df.loc[rng.choice(n, 80, replace=False), "bearing_temp_c"] = np.nan
df.loc[rng.choice(n, 40, replace=False), "pump_model"] = np.nan

In [203]:
df.info()
df

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   pump_id              2000 non-null   int64  
 1   age_years            2000 non-null   float64
 2   vibration_mm_s       2000 non-null   float64
 3   bearing_temp_c       1920 non-null   float64
 4   hours_since_service  2000 non-null   float64
 5   pump_model           1960 non-null   str    
 6   failed_7d            2000 non-null   int64  
 7   repair_ticket        2000 non-null   int64  
dtypes: float64(4), int64(3), str(1)
memory usage: 125.1 KB


,pump_id,age_years,vibration_mm_s,bearing_temp_c,hours_since_service,pump_model,failed_7d,repair_ticket
0,115,7.7,2.22,66.8,618.0,PX200,1,1
1,55,9.8,6.50,60.7,1246.0,PX100,0,0
2,99,12.7,1.59,49.4,50.0,QZ9,0,0
3,30,5.1,0.84,64.4,172.0,PX200,0,0
4,141,1.5,1.15,68.8,2854.0,PX100,0,0
...,...,...,...,...,...,...,...,...
1995,6,0.8,9.32,50.5,2279.0,PX100,0,0
1996,35,4.5,3.62,61.0,3768.0,PX100,0,0
1997,24,6.0,4.10,55.6,3357.0,PX200,0,0
1998,63,10.8,3.25,70.5,2887.0,PX200,1,1


In [208]:
print(df.failed_7d.mean(), df.failed_7d.sum())
df.pump_model.unique()
df.repair_ticket.mean()

0.129 258


np.float64(0.1385)

In [247]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

## == leakage check
TARGET = 'failed_7d'
num = df.select_dtypes(include='number').columns.drop(TARGET)
auc = {}

for c in num:
    m = df[c].notna()
    auc[c] = roc_auc_score(df.loc[m, TARGET], df.loc[m,c])

s = pd.Series(auc)
screen = pd.DataFrame({"auc": s, "str": np.maximum(s, 1-s)}).sort_values('str').round(3)
screen
df.groupby("pump_model", dropna =False)[TARGET].agg(['mean', 'count']).round(3)

# ## -- 
# # num_cols= df.select_dtypes(include = 'number').drop(['repair_ticket', 'failed_7d'], axis =1).columns.tolist()
# num_cols = ["age_years", "vibration_mm_s", "bearing_temp_c", "hours_since_service"]
# cat_cols = ['pump_model']

# X, y = df[num_cols + cat_cols], df['failed_7d']

# X_tr, X_te, y_tr, y_te = train_test_split(
#     X, y, stratify=y, test_size=0.2, random_state=00
# )

# num_pipe = Pipeline([
#     ('imp', SimpleImputer(strategy='median')),
#     ('sc', StandardScaler()),
# ])

# cat_pipe = Pipeline([
#     ('imp', SimpleImputer(strategy='most_frequent')),
#     ('ohe', OneHotEncoder())
# ])

# pre = ColumnTransformer([
#     ('num', num_pipe, num_cols),
#     ('cat', cat_pipe, cat_cols),
# ])

# dum = DummyClassifier()
# logit = Pipeline([
#     ('pre', pre),
#     ('model', LogisticRegression())
# ])
# hgb = Pipeline([
#     ('pre', pre),
#     ('model', HistGradientBoostingClassifier())
# ])

# res = {
#     'model': [],
#     'train_roc': [],
#     'test_roc': [],
#     'test_pr_auc': [],
# }

# for name, mod in [('dummy', dum), ('logit', logit), ('hgb', hgb)]:
#     mod.fit(X_tr, y_tr)
#     tr_proba = mod.predict_proba(X_tr)[:, 1]
#     te_proba = mod.predict_proba(X_te)[:, 1]

#     res['model'].append(name)
#     res['train_roc'].append(roc_auc_score(y_tr, tr_proba))
#     res['test_roc'].append(roc_auc_score(y_te, te_proba))
#     res['test_pr_auc'].append(average_precision_score(y_te, te_proba))

# pd.DataFrame(res)


,mean,count
pump_model,,
PX100,0.102,950
PX200,0.124,708
QZ9,0.225,302
NaN,0.125,40


In [222]:
y_tr.sum()

np.int64(206)

In [221]:
52 / 400 

0.13

In [224]:
206/1600

0.12875

In [248]:
import numpy as np, pandas as pd
rng = np.random.default_rng(33)
dates = pd.date_range("2025-01-01", "2026-06-30", freq="D")
t = np.arange(len(dates))
weekly = np.array([0, 25, 35, 30, 15, -70, -110])[dates.dayofweek]
y = 1000 + 0.08*t + weekly + 40*np.sin(2*np.pi*dates.dayofyear/365.25) + rng.normal(0, 20, len(dates))
ts = pd.DataFrame({"date": dates, "throughput_t": y.round(1)})
ts = ts.drop(rng.choice(len(ts), 10, replace=False)).sample(frac=1, random_state=0).reset_index(drop=True)

In [287]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.base import clone

d = ts.copy().sort_values('date').set_index('date').asfreq('D')
y = d['throughput_t']
d['lag7'] = y.shift(7)
d["lag14"]      = y.shift(14)                                  # same weekday 2 weeks ago
d["roll7_lag7"] = y.shift(7).rolling(7, min_periods=4).mean()  # average level of the week before last

d['dow'] = d.index.dayofweek
doy=d.index.day_of_year
d['doy_sin'] = np.sin(2*np.pi * doy / 365.25)
d["doy_cos"] = np.cos(2*np.pi*doy/365.25)  
d['t'] = (d.index - d.index[0]).days

## prep for train & test set
d = d.dropna(subset='throughput_t')

tr = d[d.index <  "2026-05-01"]           
te = d[d.index >= "2026-05-01"]    

##
ok = te['lag7'].notna()
print("seasonal naive MAE:", round(mean_absolute_error(te.loc[ok, "throughput_t"], te.loc[ok, "lag7"]), 2))  # 19.44


## 
NUM = ["lag7", "lag14", "roll7_lag7", "doy_sin", "doy_cos", "t"]
CAT = ["dow"]                                 # weekday as a category, not a 

num_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler())
])

cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore')),
])

pre = ColumnTransformer([
    ('num', num_pipe, NUM), ('cat', cat_pipe, CAT)
])

models = {
    'ridge': Pipeline([('pre', pre), ('m', Ridge(alpha=1.0))]),
    "hgb":   Pipeline([("pre", clone(pre)), ("m", HistGradientBoostingRegressor(random_state=0))]),
}

tscv = TimeSeriesSplit(n_splits=5)

for name, m in models.items():
    cv = -cross_val_score(m, tr[NUM + CAT], tr["throughput_t"], cv=tscv,
                      scoring="neg_mean_absolute_error")
    m.fit(tr[NUM + CAT], tr["throughput_t"])
    test_mae = mean_absolute_error(te["throughput_t"], m.predict(te[NUM + CAT]))
    print(f"{name}: CV {cv.mean():.2f} ± {cv.std():.2f} | test {test_mae:.2f}")

seasonal naive MAE: 19.44
ridge: CV 18.16 ± 3.77 | test 15.43
hgb: CV 23.41 ± 2.54 | test 18.28


In [281]:
te

,throughput_t,lag7,lag14,roll7_lag7,dow,doy_sin,doy_cos,t
date,,,,,,,,
2026-05-01,1086.1,1096.9,1096.1,1064.414286,4,0.872404,-0.488785,485
2026-05-02,999.9,990.8,1008.1,1061.942857,5,0.863867,-0.503720,486
2026-05-03,977.6,994.0,976.7,1064.414286,6,0.855075,-0.518505,487
2026-05-04,1073.5,1113.5,1063.5,1071.557143,0,0.846029,-0.533137,488
2026-05-05,1112.3,1111.1,1098.9,1073.300000,1,0.836733,-0.547611,489
...,...,...,...,...,...,...,...,...
2026-06-26,1081.3,1084.1,1090.8,1045.614286,4,0.096613,-0.995322,541
2026-06-27,963.7,974.8,969.0,1046.442857,5,0.079477,-0.996837,542
2026-06-28,918.9,921.6,965.1,1040.228571,6,0.062318,-0.998056,543


In [145]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_val_score

NUM, CAT = ["tonnage", "temp_c"], ["ore_type", "shift"]
X, y = df.drop(columns="energy_mwh"), df["energy_mwh"]

num_pipe = Pipeline([("imp", SimpleImputer(strategy='median')), ("sc", StandardScaler())])
cat_pipe = Pipeline([("imp", SimpleImputer(strategy='most_frequent')),
                     ("ohe", OneHotEncoder(handle_unknown='ignore'))])
pre  = ColumnTransformer([("num", num_pipe, NUM), ("cat", cat_pipe, CAT)])
pipe = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
# dum_pip = Pipeline([("pre", pre), ("model", DummyRegressor())])

cv = KFold(5, shuffle=True, random_state=0)
# loop over [("dummy", DummyRegressor()), ("ridge", pipe)] → cross_val_score(..., scoring="neg_mean_absolute_error")

for name, m in [("dummy", DummyRegressor()), ("ridge", pipe)]:
    mae = -cross_val_score(m, X, y, cv = cv, scoring='neg_mean_absolute_error')
    print(f"{name:6s} MAE: {mae.mean():.3f} ± {mae.std():.3f}")

pipe.fit(X, y)
print(pipe.predict(new_rows).round(2))
print(pipe.named_steps["pre"].get_feature_names_out())

dummy  MAE: 2.074 ± 0.238
ridge  MAE: 0.830 ± 0.059
[23.24 21.5 ]
['num__tonnage' 'num__temp_c' 'cat__ore_type_mixed' 'cat__ore_type_oxide'
 'cat__ore_type_sulfide' 'cat__shift_Day' 'cat__shift_Night']


In [143]:
cv

KFold(n_splits=5, random_state=0, shuffle=True)